<a href="https://colab.research.google.com/github/Yamato603/colab-Pix2Pix/blob/main/Pix2Pix_Macular_Hole2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix

Cloning into 'pytorch-CycleGAN-and-pix2pix'...
remote: Enumerating objects: 2619, done.
remote: Total 2619 (delta 0), reused 0 (delta 0), pack-reused 2619 (from 1)
Receiving objects: 100% (2619/2619), 8.24 MiB | 20.13 MiB/s, done.
Resolving deltas: 100% (1654/1654), done.


In [2]:
import os
os.chdir('pytorch-CycleGAN-and-pix2pix/')

In [3]:
# 必要な3つの追加ツール（dominate, visdom, wandb）を直接インストールする
!pip install dominate visdom wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 73.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for visdom: filename=visdom-0.2.4-py3-none-any.whl size=1408195 sha256=0ff1639365880b5df3e97bd7c7dcb61183f1de7354ed8e5d5ecd98b964b522aa
  Stored in directory: /root/.cache/pip/wheels/37/6c/38/64eeaa310e325aacda723e6df1f79ab5e9f31ba195264e04a8
Successfully built visdom


In [4]:
# スクリプトの先頭にインポート文を挿入して修正
!sed -i '1i from pathlib import Path' /content/pytorch-CycleGAN-and-pix2pix/datasets/combine_A_and_B.py

print("スクリプトの修正が完了しました！")

スクリプトの修正が完了しました！


In [6]:
import os
import glob
import random
from PIL import Image
from google.colab import drive
import torchvision.transforms.functional as TF

# =========================================================
# 1. Google Driveの連携とZIPファイルの解凍設定
# =========================================================
print("--- Step 1: Google Driveを連携します ---")
drive.mount('/content/drive')

# マイドライブにアップロードされたZIPファイルのパス
zip_path_pre = '/content/drive/MyDrive/Pre_pick256.zip'
zip_path_pos = '/content/drive/MyDrive/Pos_pick256.zip'

# 作業用およびデータセット出力先のフォルダ定義
raw_base_dir = '/content/pytorch-CycleGAN-and-pix2pix/datasets/oct_raw'
out_dir = '/content/pytorch-CycleGAN-and-pix2pix/datasets/oct_pix2pix_256'

# --- データ拡張（水増し）の設定 ---
# 学習用（train）の元画像1枚あたり、追加で何枚の拡張バリエーションを生成するか
# （例: 5にすると、元画像1枚から5枚の異なる変形画像が作られ、データ量が6倍になります）
AUG_MULTIPLIER = 5

# 既存の古い一時フォルダをリセット
os.system(f'rm -rf "{raw_base_dir}"')
os.system(f'rm -rf "{out_dir}"')
os.makedirs(f"{raw_base_dir}/Pre", exist_ok=True)
os.makedirs(f"{raw_base_dir}/Pos", exist_ok=True)

print("--- Step 2: ZIPファイルを解凍しています ---")
os.system(f'unzip -q -o "{zip_path_pre}" -d "{raw_base_dir}/Pre"')
os.system(f'unzip -q -o "{zip_path_pos}" -d "{raw_base_dir}/Pos"')
print("解凍作業が完了しました。\n")


# =========================================================
# 2. 画像のペアリング・データ拡張・左右結合処理
# =========================================================
print("--- Step 3: 画像ペアの探索とデータ拡張・結合を開始します ---")

# 解凍されたフォルダ内からすべての画像ファイルパスを取得
pre_files = glob.glob(os.path.join(raw_base_dir, "Pre", "**", "*.*"), recursive=True)
pos_files = glob.glob(os.path.join(raw_base_dir, "Pos", "**", "*.*"), recursive=True)

# 術後(Pos)の画像ファイルをファイル名ベースで辞書化（検索を高速化するため）
pos_dict = {os.path.basename(f): f for f in pos_files if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))}

pairs = []
for pre_path in pre_files:
    if not pre_path.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
        continue

    filename = os.path.basename(pre_path)
    post_path = None

    # 完全一致でのペアリング
    if filename in pos_dict:
        post_path = pos_dict[filename]
    else:
        # ★マークの有無による表記揺れを補正して再探索
        alt_no_star = filename.replace('★', '')
        alt_with_star = '★' + filename

        if alt_no_star in pos_dict:
            post_path = pos_dict[alt_no_star]
        elif alt_with_star in pos_dict:
            post_path = pos_dict[alt_with_star]

    if post_path:
        pairs.append((pre_path, post_path))
    else:
        print(f"[確認] 対応する術後画像が見つからないためスキップします: {filename}")

# 全体のペアをランダムにシャッフルし、学習80%、検証10%、テスト10%に分割
random.seed(42)
random.shuffle(pairs)

total = len(pairs)
if total == 0:
    print("【エラー】有効な画像ペアが見つかりませんでした。ZIPファイル内の構造や画像形式を再確認してください。")
else:
    train_split = int(total * 0.8)
    val_split = int(total * 0.9)

    splits = {
        'train': pairs[:train_split],
        'val': pairs[train_split:val_split],
        'test': pairs[val_split:]
    }

    print(f"合計 {total} ペアのオリジナルデータからデータセットを構築します。")

    count = 0
    for split_name, pair_list in splits.items():
        split_dir = os.path.join(out_dir, split_name)
        os.makedirs(split_dir, exist_ok=True)

        split_count = 0
        for pre_path, post_path in pair_list:
            img_a = Image.open(pre_path).convert('RGB')
            img_b = Image.open(post_path).convert('RGB')

            w_a, h_a = img_a.size
            w_b, h_b = img_b.size

            # 万が一256x256ではない画像が混入していた場合の自動リサイズ保険
            if w_a != 256 or h_a != 256: img_a = img_a.resize((256, 256), Image.Resampling.LANCZOS)
            if w_b != 256 or h_b != 256: img_b = img_b.resize((256, 256), Image.Resampling.LANCZOS)

            # ---------------------------------------------------------
            # A: オリジナル画像ペアの左右連結 (512x256) と保存
            # ---------------------------------------------------------
            img_ab = Image.new('RGB', (256 * 2, 256))
            img_ab.paste(img_a, (0, 0))
            img_ab.paste(img_b, (256, 0))

            out_filename = f"{count:05d}.jpg"
            img_ab.save(os.path.join(split_dir, out_filename))
            count += 1
            split_count += 1

            # ---------------------------------------------------------
            # B: 指定された4つのデータ拡張処理（学習データのみに適用）
            # ---------------------------------------------------------
            if split_name == 'train' and AUG_MULTIPLIER > 0:
                for m in range(AUG_MULTIPLIER):
                    # 4つの幾何学変換パラメータをランダムに決定
                    # 医療画像の解剖学的構造を破綻させない微小な範囲に制限しています
                    angle = random.uniform(-5.0, 5.0)         # ① 角度回転 (-5度 〜 +5度)
                    translate = [
                        int(random.uniform(-15, 15)),         # ③ 左右にずらす (±15ピクセル)
                        int(random.uniform(-15, 15))          # ② 上下にずらす (±15ピクセル)
                    ]
                    scale = random.uniform(0.92, 1.08)        # ④ 拡大縮小 (0.92倍 〜 1.08倍)
                    shear = 0.0                               # せん断変形（今回は歪み防止のため0に固定）

                    # 術前(A)と術後(B)に「全く同じパラメータ」を一対一で同期適用
                    # 変形によってはみ出た境界の余白は黒(fill=0)で埋めます
                    aug_a = TF.affine(img_a, angle=angle, translate=translate, scale=scale, shear=shear, fill=0)
                    aug_b = TF.affine(img_b, angle=angle, translate=translate, scale=scale, shear=shear, fill=0)

                    # 拡張を適用したAとBを左右に連結
                    aug_ab = Image.new('RGB', (256 * 2, 256))
                    aug_ab.paste(aug_a, (0, 0))
                    aug_ab.paste(aug_b, (256, 0))

                    aug_filename = f"{count:05d}_aug_{m}.jpg"
                    aug_ab.save(os.path.join(split_dir, aug_filename))
                    count += 1
                    split_count += 1

        print(f" ✔ 【{split_name}】: 合計 {split_count} 枚を保存しました（データ拡張分を含む）")

    print("\n=========================================================")
    print(" すべてのデータ構築および指定のデータ拡張が完了しました。")
    print(f" 出力先フォルダ: {out_dir}")
    print("=========================================================")

--- Step 1: Google Driveを連携します ---
Mounted at /content/drive
--- Step 2: ZIPファイルを解凍しています ---
解凍作業が完了しました。

--- Step 3: 画像ペアの探索とデータ拡張・結合を開始します ---
合計 182 ペアのオリジナルデータからデータセットを構築します。
 ✔ 【train】: 合計 870 枚を保存しました（データ拡張分を含む）
 ✔ 【val】: 合計 18 枚を保存しました（データ拡張分を含む）
 ✔ 【test】: 合計 19 枚を保存しました（データ拡張分を含む）

 すべてのデータ構築および指定のデータ拡張が完了しました。
 出力先フォルダ: /content/pytorch-CycleGAN-and-pix2pix/datasets/oct_pix2pix_256


In [7]:
%cd /content/pytorch-CycleGAN-and-pix2pix

!python train.py \
  --dataroot ./datasets/oct_pix2pix_256 \
  --name oct_macular_hole \
  --model pix2pix \
  --direction AtoB \
  --preprocess none \
  --batch_size 1 \
  --input_nc 1 \
  --output_nc 1 \
  --n_epochs 50 \
  --n_epochs_decay 50

/content
----------------- Options ---------------
               batch_size: 1                             
                    beta1: 0.5                           
          checkpoints_dir: ./checkpoints                 
           continue_train: False                         
                crop_size: 256                           
                 dataroot: ./datasets/oct_pix2pix_256    	[default: None]
             dataset_mode: aligned                       
                direction: AtoB                          
             display_freq: 400                           
          display_winsize: 256                           
                    epoch: latest                        
              epoch_count: 1                             
                 gan_mode: vanilla                       
                init_gain: 0.02                          
                init_type: normal                        
                 input_nc: 1                             	[defa

In [8]:
# 作業フォルダへ移動
%cd /content/pytorch-CycleGAN-and-pix2pix

!python test.py \
  --dataroot ./datasets/oct_pix2pix_256 \
  --name oct_macular_hole \
  --model pix2pix \
  --direction AtoB \
  --preprocess none \
  --input_nc 1 \
  --output_nc 1 \
  --results_dir ./results/

/content
----------------- Options ---------------
             aspect_ratio: 1.0                           
               batch_size: 1                             
          checkpoints_dir: ./checkpoints                 
                crop_size: 256                           
                 dataroot: ./datasets/oct_pix2pix_256    	[default: None]
             dataset_mode: aligned                       
                direction: AtoB                          
          display_winsize: 256                           
                    epoch: latest                        
                     eval: False                         
                init_gain: 0.02                          
                init_type: normal                        
                 input_nc: 1                             	[default: 3]
                  isTrain: False                         	[default: None]
                load_iter: 0                             	[default: 0]
                load_

In [9]:
# 1. Google Drive内に保存用フォルダを作成（既にある場合は自動でスキップされます）
#!mkdir -p /content/drive/MyDrive/OCT_Weights

# 2. Pix2Pixのチェックポイントフォルダをご自身のGoogle Driveへ丸ごとコピー
#!cp -r /content/pytorch-CycleGAN-and-pix2pix/checkpoints/oct_macular_hole /content/drive/MyDrive/OCT_Weights/

#print("Google Driveへの重みのバックアップが完了しました！")
